In [1]:
import sys
#!{sys.executable} -m pip install gseapy
#/Users/allenma/.local/bin

In [2]:
import pandas as pd
import gseapy
import numpy as np
import os

In [5]:
AREAindir="/scratch/Shares/dowell/temp/ChrisO/PSEA/AREA_fast/output/AREA_output/all_chrom_comorbidthreshold0/"
AREAresultsfile = AREAindir+"area_scores_20250808-192325.adjpval.csv"
outdir="/Users/allenma/area_runs/AREA_2025/first_GSEA/"


In [6]:
indir="/Shares/down/public/INLCUDE_2024/AREA_paper_2025/output_data/AREA_output/"
#
allfileHP = indir+"400_WG_filterednormcounts_MONDO_area_scores_20260106-122018.adjpval.csv" #not right, need hp soon
T21fileHP= indir+"254T21_WG_filterednormcounts_MONDO_area_scores_20260109-142327.adjpval.csv" #not right, need hp soon
allfileMONDO= indir+"400_WG_filterednormcounts_MONDO_area_scores_20260106-122018.adjpval.csv"
T21fileMONDO = indir+"254T21_WG_filterednormcounts_MONDO_area_scores_20260109-142327.adjpval.csv"
geneindir = "/Shares/down/public/INLCUDE_2024/kallisto_20241030/selfannoated/"
genefile = geneindir+"genes.csv"
AREAresultsfile = allfileMONDO

In [25]:
def rankAREAresultsfile(AREAresultsfile, genefile):
    genedf = pd.read_csv(genefile)
    switchgenename = genedf[["gene_name", "gene_id"]].copy().drop_duplicates()
    switchgenename["ranked_by"]=switchgenename["gene_id"]
    df = pd.read_csv(AREAresultsfile, low_memory=False)
    df = df.dropna().copy()
    df["pval"]=df["pval"].astype(float)
    df["NES"]=df["NES"].astype(float)
    df["rnk_score"] = np.nan
    dfno0 = df[df["pval"]!=0] 
    minpval = dfno0["pval"].min() 
    print(minpval)
    mask = df["pval"] > 0
    print(df.shape)
    df.loc[mask, "rnk_score"] = -np.log(df.loc[mask, "pval"]) * np.sign(df.loc[mask, "NES"])
    df.loc[~mask, "rnk_score"] = -np.log(minpval) * np.sign(df.loc[~mask, "NES"])
    df = df.merge(switchgenename, how="inner", on="ranked_by")
    print(df.shape)
    return df


def prerank1(df, comorbid, outdir):
    comorbid_df = df[df["boolean_attribute"]==comorbid]
    comorbid_df = comorbid_df.sort_values("rnk_score")
    comorbid_df = comorbid_df[["gene_name", "rnk_score"]]
    fn = outdir+comorbid+".rnk"
    comorbid_df = comorbid_df.dropna()
    if comorbid_df.shape[0]!=0:
        print (comorbid, comorbid_df.shape)
        comorbid_df.to_csv(outdir+comorbid+".rnk", sep="\t", header=False, index=False)
        return fn
    else:
        return fn

def prerankall(df, outdir):
    filenames = []
    comorbids = df["boolean_attribute"].unique()
    for comorbid in comorbids:
        fn = prerank1(df, comorbid, outdir)
        filenames.append(fn)
    return filenames

In [26]:
df = rankAREAresultsfile(AREAresultsfile, genefile)

1.7622841634848939e-81
(1298856, 9)
(0, 11)


In [20]:
df

,boolean_attribute,ranked_by,NES,pval,p_value_bonf,p_value_holm,p_value_BenjaminiHochberg,p_value_BenjaminiYekutieli,rnk_score,gene_name,gene_id


In [19]:
prerankall(df, outdir)

[]

In [67]:
arankfile = "/Users/allenma/area_runs/AREA_2025/first_GSEA/HP_Sleep_apnea.rnk"
readfile = pd.read_csv(arankfile, sep="\t", header=False)
readfile

TypeError: Passing a bool to header is invalid. Use header=None for no header or header=int or list-like of ints to specify the row(s) making up the column names

In [98]:

pr = gseapy.prerank(rnk="/Users/allenma/area_runs/AREA_2025/first_GSEA/HP_Sleep_apnea.rnk", gene_sets='KEGG_2016', outdir='test')


2025-08-13 16:17:12,194 [WARNING] Input gene rankings contains inf values!
/Users/allenma/.local/lib/python3.9/site-packages/gseapy/gsea.py:507: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  rankser.replace(-np.inf, method="ffill", inplace=True)
/Users/allenma/.local/lib/python3.9/site-packages/gseapy/gsea.py:508: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  rankser.replace(np.inf, method="bfill", inplace=True)
2025-08-13 16:17:12,197 [WARNING] Duplicated values found in preranked stats: 59.12% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [99]:
pr